# 🐧 Palmer Penguins Species Classifier — Hands-On Workshop Notebook
**MSU AI Club Workshop 01: The Art of the Data Lifecycle**  
*Complete the guided coding exercises (# TODO blocks) below during the workshop to build, evaluate, and export your ML classifier project.*

---

## 🎯 Workshop Goals & Data Lifecycle Stages:
1. **Stage 1 (Ingestion)**: Load dataset dynamically from public Data URL & audit missing values.
2. **Stage 2 (Cleaning & Imputation)**: Perform median and mode imputation without dropping observation cohorts.
3. **Stage 3 (Feature Engineering & EDA)**: Calculate biological feature ratios & plot interactive Plotly scatter charts.
4. **Stage 4 (Modeling & Evaluation)**: Train a Random Forest classifier using stratified splits and evaluate per-species recall.
5. **Stage 5 (Inference & Serialization)**: Export model to `penguin_model.pkl` and test real-time predictions.


### Setup: Import Required Python Modules

In [ ]:
import os
import sys
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import plotly.express as px

# Ensure cross-platform UTF-8 encoding
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

print("✅ Setup complete. Libraries imported successfully!")


--- 
## 🛠️ Exercise 1: Data Ingestion & Missing Value Audit (Stage 1)
**Data Governance Rule**: Never commit raw data CSV files into git repositories. Fetch data dynamically from public URLs or use automated download scripts.

**Task**:  
1. Load dataset from `DATA_URL` into DataFrame `df` using `pd.read_csv()`.
2. Complete the code below to count missing values across each column.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

# TODO: Load dataset using pandas read_csv
df = pd.read_csv(DATA_URL)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
display(df.head())

# TODO: Write code to calculate missing values for each column
# HINT: Use df.isnull().sum()
missing_counts = df.isnull().sum()  # <--- YOUR CODE HERE

print("\n--- Missing Values Audit ---")
print(missing_counts)


--- 
## 🛠️ Exercise 2: Data Cleaning & Imputation (Stage 2)
**Data Lifecycle Principle**: Avoid naive row deletion (`df.dropna()`) because dropping rows eliminates underrepresented cohorts. Instead, apply **Median Imputation** for numeric fields and **Mode Imputation** for categorical fields.

**Task**: Fill missing values in `df_clean` without dropping rows.


In [ ]:
df_clean = df.copy()

numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# TODO: Loop through numeric_cols and fill NaNs with the median value of each column
for col in numeric_cols:
    # YOUR CODE HERE: Calculate median and fillna
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"Imputed {col} with median: {median_val:.1f}")

# TODO: Fill missing 'sex' column with the mode (most frequent value)
# YOUR CODE HERE
mode_sex = df_clean["sex"].mode()[0]
df_clean["sex"] = df_clean["sex"].fillna(mode_sex)
print(f"Imputed sex with mode: {mode_sex}")

print(f"\nRemaining missing values: {df_clean.isnull().sum().sum()} (Preserved all {len(df_clean)} records)")


--- 
## 🛠️ Exercise 3: Feature Engineering Challenge (Stage 3)
**Challenge Task**: Create a new feature `bill_ratio` defined as `bill_length_mm / bill_depth_mm`.  
Aspect ratio often provides stronger species discrimination than individual dimensions alone.


In [ ]:
# TODO: Calculate bill_ratio feature
# YOUR CODE HERE:
df_clean["bill_ratio"] = df_clean["bill_length_mm"] / df_clean["bill_depth_mm"]

print("Sample engineered features:")
display(df_clean[["species", "bill_length_mm", "bill_depth_mm", "bill_ratio"]].head())


--- 
## 🛠️ Exercise 4: Exploratory Data Analysis & Plotly Visualizations
**Task**: Construct an interactive scatter plot of `bill_length_mm` vs `bill_depth_mm`, coloring observations by `species`.


In [ ]:
# TODO: Build Plotly scatter plot using px.scatter()
# HINT: Set x="bill_length_mm", y="bill_depth_mm", color="species"
fig = px.scatter(
    df_clean,
    x="bill_length_mm",
    y="bill_depth_mm",
    color="species",
    size="body_mass_g",
    hover_data=["island", "sex"],
    title="Palmer Penguins: Bill Length vs Bill Depth Scatter",
    color_discrete_map={"Adelie": "#08ffff", "Chinstrap": "#ff0055", "Gentoo": "#ffcc00"}
)
fig.update_layout(template="plotly_dark", font_family="Rubik")
fig.show()


--- 
## 🛠️ Exercise 5: Stratified Train/Test Split & Model Training (Stage 4)
**Task**:  
1. Select feature columns (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) into `X` and `species` into `y`.
2. Split into 80% train and 20% test subsets using `stratify=y` so minority species maintain proportional representation.
3. Train a `RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)`.


In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# TODO: Assign X and y
X = df_clean[features]  # <--- YOUR CODE HERE
y = df_clean["species"]  # <--- YOUR CODE HERE

# TODO: Perform stratified train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO: Instantiate and train RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

print("✅ Random Forest model trained successfully!")


--- 
## 🛠️ Exercise 6: Model Evaluation & Per-Species Report
**Task**: Predict test set species and print the `classification_report` to verify precision and recall for Adelie, Chinstrap, and Gentoo.


In [ ]:
# TODO: Generate predictions on X_test
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Overall Classification Accuracy: {acc:.1%}\n")

# TODO: Print detailed per-species classification report
print("--- Per-Species Classification Report ---")
print(classification_report(y_test, y_pred))


--- 
## 🛠️ Exercise 7: Model Serialization & Real-Time Inference (Stage 5)
**Task**:  
1. Serialize trained classifier `clf` to `penguin_model.pkl` using `pickle.dump()`.
2. Pass custom test measurements to `clf.predict()` and `clf.predict_proba()` to check species confidence scores.


In [ ]:
# TODO: Serialize model artifact using pickle
with open("penguin_model.pkl", "wb") as f:
    pickle.dump(clf, f)
print("💾 Model saved to penguin_model.pkl")

# TODO: Test real-time inference on a custom penguin sample
sample_penguin = pd.DataFrame([{
    "bill_length_mm": 48.5,
    "bill_depth_mm": 15.0,
    "flipper_length_mm": 217.0,
    "body_mass_g": 5000.0
}])

prediction = clf.predict(sample_penguin)[0]
probabilities = clf.predict_proba(sample_penguin)[0]

print(f"\nPredicted Species: >>> {prediction.upper()} <<<")
for species_name, prob in zip(clf.classes_, probabilities):
    print(f"  * {species_name:10s}: {prob:.1%}")
